# Fabric Gold Packaging And Deployment



This notebook packages the Gold lakehouse objects from workspace LoanDemoEnv for GitHub transport and optionally deploys them into a target lakehouse that you specify.



If the target lakehouse does not exist yet, the notebook can create it through Fabric REST before loading the exported Gold tables.



Scope is restricted to the ontology requested as `AnalyceLoanV3`, resolved in Fabric as `AnalyzeLoanOnV3`.

In [ ]:
from pyspark.sql import Row



from pyspark.sql.utils import AnalysisException



import json



import requests







tenant_id = "68306248-6766-4cfe-8693-8b99abf3cba6"



source_workspace_name = "LoanDemoEnv"



source_workspace_id = "031bbc7e-6f0e-4dd0-85a0-fee11d03fb0a"



requested_ontology_name = "AnalyceLoanV3"



resolved_ontology_name = "AnalyzeLoanOnV3"



resolved_ontology_id = "ef14041b-60e3-412c-9084-eee9fedb76e2"



source_ontology_lakehouse = "AnalyzeLoanOnV3_lh_ef14041b60e3412c9084eee9fedb76e2"



source_ontology_lakehouse_id = "6834ed2a-e76b-44a6-a316-19c11c67a680"



source_gold_lakehouse = "Gold"



source_gold_lakehouse_id = "2cfc458b-236e-4d9b-b5fd-394c07bc1b86"



source_gold_sql_endpoint_id = "1d313e12-e9b1-4f9c-9d68-0644a3f96df4"



source_schema = "dbo"







deployment_notebooks = [



    {"name": "loadgold", "id": "9ba10749-f3e3-4115-84ce-b6728bbd347d", "role": "Builds dimensions and fact tables in Gold"},



    {"name": "GoldImprovments", "id": "b19e5a56-51b3-4a7e-adeb-df517b365d47", "role": "Applies fixes including V2 entities used by ontology"},



]







gold_objects = [



    {"load_order": 1, "object_name": "dim_loan", "object_type": "dimension", "required": True, "source_notebook": "GoldImprovments/loadgold", "purpose": "Loan master dimension", "candidates": ["dim_loan"]},



    {"load_order": 2, "object_name": "dim_prev_application", "object_type": "dimension", "required": True, "source_notebook": "loadgold", "purpose": "Previous application dimension", "candidates": ["dim_prev_application"]},



    {"load_order": 3, "object_name": "dim_relative_month", "object_type": "dimension", "required": True, "source_notebook": "loadgold", "purpose": "Relative month dimension", "candidates": ["dim_relative_month"]},



    {"load_order": 4, "object_name": "dim_dpd_bucket_v2", "object_type": "dimension", "required": True, "source_notebook": "GoldImprovments/loadgold", "purpose": "DPD bucket V2 dimension bound to ontology", "candidates": ["dim_dpd_bucket_v2", "dim_dpd_bucketV2", "dim_dpdV2", "dim_dpd_v2", "dim_dpd_bucket"]},



    {"load_order": 5, "object_name": "fact_pos_cash_monthly_balance_v2", "object_type": "fact", "required": True, "source_notebook": "loadgold", "purpose": "POS cash monthly balance V2 fact bound to ontology", "candidates": ["fact_pos_cash_monthly_balance_v2", "fact_pos_cash_monthly_balanceV2", "fact_credit_card_monthly_balanceV2", "fact_credit_card_monthly_balance_v2", "fact_credit_card_monthly_balance"]},



]







required_for_ontology = [



    "dim_loan",



    "dim_prev_application",



    "dim_relative_month",



    "dim_dpd_bucket_v2",



    "fact_pos_cash_monthly_balance_v2",



]







export_root = "Files/github_portable/gold"



manifest_root = f"{export_root}/manifests"



csv_root = f"{export_root}/csv"



count_rows = False







deploy_to_target = True



target_workspace_name = source_workspace_name



target_workspace_id = source_workspace_id



target_gold_lakehouse = "Gold_AnalyzeLoanV3_Target"



target_gold_lakehouse_id = "49b3f9e2-0aa8-457c-bd6f-ea87a2cf15e3"



target_schema = "dbo"



create_target_lakehouse_if_missing = True



target_lakehouse_enable_schemas = True



# Ontology deployment controls

restore_ontology_to_target = True

target_ontology_name = resolved_ontology_name

target_ontology_id = None

overwrite_target_ontology_definition = True



fabric_token_override = None







print(f"Packaging scope: {resolved_ontology_name} -> {source_gold_lakehouse}")



print(f"Export root: /lakehouse/default/{export_root}")



print(f"Deploy enabled: {deploy_to_target}")



print(f"Target workspace: {target_workspace_name} ({target_workspace_id})")



print(f"Target lakehouse: {target_gold_lakehouse} ({target_gold_lakehouse_id})")



print(f"Create target lakehouse if missing: {create_target_lakehouse_if_missing}")



print(f"Restore ontology to target: {restore_ontology_to_target}")


In [ ]:
def qualified_name(object_name: str, lakehouse_name: str = source_gold_lakehouse, schema_name: str = source_schema) -> str:

    return f"{lakehouse_name}.{schema_name}.{object_name}"



def onelake_path(relative_path: str) -> str:

    return f"/lakehouse/default/{relative_path.strip('/')}"



def write_single_csv(dataframe, relative_path: str) -> str:

    full_path = onelake_path(relative_path)

    dataframe.coalesce(1).write.mode("overwrite").option("header", "true").csv(full_path)

    return full_path



def read_table_safe(table_name: str):

    try:

        return spark.read.table(table_name)

    except AnalysisException:

        return None



def object_rows():

    rows = []

    for item in gold_objects:

        rows.append(Row(

            load_order=item["load_order"],

            object_name=item["object_name"],

            object_type=item["object_type"],

            source_table=qualified_name(item["object_name"]),

            required=item["required"],

            source_notebook=item["source_notebook"],

            purpose=item["purpose"],

        ))

    return rows



runtime_notebookutils = globals().get("notebookutils")



def get_fabric_token():

    if fabric_token_override:

        return fabric_token_override



    if runtime_notebookutils is None:

        return None



    candidates = []



    try:

        candidates.append(runtime_notebookutils.credentials.getToken("pbi"))

    except Exception:

        pass



    try:

        candidates.append(runtime_notebookutils.credentials.getToken("https://api.fabric.microsoft.com"))

    except Exception:

        pass



    for candidate in candidates:

        if candidate:

            return candidate



    return None



def fabric_headers(token: str):

    return {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}



def list_workspace_items(workspace_id: str, token: str):

    response = requests.get(

        f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/items",

        headers=fabric_headers(token),

        timeout=60,

    )

    response.raise_for_status()

    return response.json().get("value", [])



def resolve_target_workspace_id() -> str:

    return target_workspace_id or source_workspace_id



def find_target_lakehouse(workspace_id: str, token: str):

    items = list_workspace_items(workspace_id, token)

    for item in items:

        if item.get("type") == "Lakehouse" and item.get("displayName") == target_gold_lakehouse:

            return item

    return None



def create_target_lakehouse(workspace_id: str, token: str):

    payload = {

        "displayName": target_gold_lakehouse,

        "type": "Lakehouse",

        "creationPayload": {

            "enableSchemas": target_lakehouse_enable_schemas

        }

    }



    response = requests.post(

        f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/items",

        headers=fabric_headers(token),

        data=json.dumps(payload),

        timeout=60,

    )

    response.raise_for_status()

    return response.json()



def ensure_target_lakehouse():

    workspace_id = resolve_target_workspace_id()



    if target_gold_lakehouse_id:

        return {

            "id": target_gold_lakehouse_id,

            "displayName": target_gold_lakehouse,

            "workspaceId": workspace_id,

            "created": False,

            "validatedBy": "parameter"

        }



    token = get_fabric_token()

    if not token:

        if create_target_lakehouse_if_missing:

            raise ValueError(

                "A Fabric token is required to validate or create the target lakehouse. "

                "Set fabric_token_override or run this notebook where notebookutils.credentials can issue a Fabric token."

            )

        return {

            "id": None,

            "displayName": target_gold_lakehouse,

            "workspaceId": workspace_id,

            "created": False,

            "validatedBy": "name-only"

        }



    existing = find_target_lakehouse(workspace_id, token)

    if existing:

        return {

            "id": existing.get("id"),

            "displayName": existing.get("displayName"),

            "workspaceId": workspace_id,

            "created": False,

            "validatedBy": "rest"

        }



    if not create_target_lakehouse_if_missing:

        raise ValueError(

            f"Target lakehouse '{target_gold_lakehouse}' was not found in workspace {workspace_id}. "

            "Set create_target_lakehouse_if_missing = True to create it automatically."

        )



    created = create_target_lakehouse(workspace_id, token)

    return {

        "id": created.get("id"),

        "displayName": created.get("displayName", target_gold_lakehouse),

        "workspaceId": workspace_id,

        "created": True,

        "validatedBy": "rest"

    }


In [ ]:
# Resolve physical source table names using candidate patterns (important for V2 variants).

def _normalize_name(value: str) -> str:

    return "".join(ch for ch in value.lower() if ch.isalnum())



def _list_source_tables():

    return [

        row["tableName"]

        for row in spark.sql(f"SHOW TABLES IN {source_gold_lakehouse}.{source_schema}").select("tableName").collect()

    ]



def _resolve_source_name(item: dict, available_tables):

    lookup = {_normalize_name(t): t for t in available_tables}

    for candidate in item.get("candidates", [item["object_name"]]):

        key = _normalize_name(candidate)

        if key in lookup:

            return lookup[key]



    normalized_object = _normalize_name(item["object_name"])

    if "v2" in normalized_object:

        for table_name in available_tables:

            normalized_table = _normalize_name(table_name)

            if "v2" not in normalized_table:

                continue

            if "dpd" in normalized_object and "dpd" in normalized_table:

                return table_name

            if "snapshot" in normalized_object and "snapshot" in normalized_table:

                return table_name



    return None



available_source_tables = _list_source_tables()

resolved_gold_objects = []

missing_required_resolved = []

for item in gold_objects:

    resolved_name = _resolve_source_name(item, available_source_tables)

    resolved_item = dict(item)

    resolved_item["resolved_object_name"] = resolved_name

    resolved_gold_objects.append(resolved_item)

    if item["required"] and not resolved_name:

        missing_required_resolved.append(item["object_name"])



if missing_required_resolved:

    raise ValueError(

        f"Missing required source objects after V2 resolution: {missing_required_resolved}. Available source tables: {sorted(available_source_tables)}"

    )



def object_rows():

    rows = []

    for item in resolved_gold_objects:

        rows.append(Row(

            load_order=item["load_order"],

            object_name=item["object_name"],

            object_type=item["object_type"],

            source_table=qualified_name(item["resolved_object_name"]),

            required=item["required"],

            source_notebook=item["source_notebook"],

            purpose=item["purpose"],

        ))

    return rows



display(spark.createDataFrame([Row(object_name=i["object_name"], resolved_object_name=i["resolved_object_name"]) for i in resolved_gold_objects]))


In [ ]:
context_df = spark.createDataFrame([Row(

    tenant_id=tenant_id,

    source_workspace_name=source_workspace_name,

    source_workspace_id=source_workspace_id,

    requested_ontology_name=requested_ontology_name,

    resolved_ontology_name=resolved_ontology_name,

    resolved_ontology_id=resolved_ontology_id,

    source_ontology_lakehouse=source_ontology_lakehouse,

    source_ontology_lakehouse_id=source_ontology_lakehouse_id,

    source_gold_lakehouse=source_gold_lakehouse,

    source_gold_lakehouse_id=source_gold_lakehouse_id,

    source_gold_sql_endpoint_id=source_gold_sql_endpoint_id,

    export_root=export_root,

    manifest_root=manifest_root,

    csv_root=csv_root,

    deploy_to_target=deploy_to_target,

    target_workspace_name=target_workspace_name,

    target_workspace_id=target_workspace_id,

    target_gold_lakehouse=target_gold_lakehouse,

    target_gold_lakehouse_id=target_gold_lakehouse_id,

    create_target_lakehouse_if_missing=create_target_lakehouse_if_missing,

)])



ontology_scope_df = spark.createDataFrame([Row(

    requested_name=requested_ontology_name,

    resolved_name=resolved_ontology_name,

    ontology_id=resolved_ontology_id,

    ontology_lakehouse=source_ontology_lakehouse,

    ontology_lakehouse_id=source_ontology_lakehouse_id,

    included_in_package=True,

    excluded_other_ontologies=True,

)])



objects_df = spark.createDataFrame(object_rows())

notebooks_df = spark.createDataFrame([Row(**item) for item in deployment_notebooks])



write_single_csv(context_df, f"{manifest_root}/package_context")

write_single_csv(ontology_scope_df, f"{manifest_root}/ontology_scope")

write_single_csv(objects_df.orderBy("load_order"), f"{manifest_root}/gold_objects")

write_single_csv(notebooks_df.orderBy("name"), f"{manifest_root}/gold_notebooks")

display(objects_df.orderBy("load_order"))


In [ ]:
status_rows = []

column_rows = []

missing_required = []



for item in resolved_gold_objects:

    source_name = item.get("resolved_object_name") or item["object_name"]

    table_name = qualified_name(source_name)

    df = read_table_safe(table_name)

    is_present = df is not None

    row_count = None

    column_count = 0



    if is_present:

        column_count = len(df.columns)

        if count_rows:

            row_count = df.count()

        for ordinal, field in enumerate(df.schema.fields, start=1):

            column_rows.append(Row(

                object_name=item["object_name"],

                column_ordinal=ordinal,

                column_name=field.name,

                data_type=field.dataType.simpleString(),

                nullable=field.nullable,

            ))

    elif item["required"]:

        missing_required.append(table_name)



    status_rows.append(Row(

        object_name=item["object_name"],

        source_table=table_name,

        required=item["required"],

        present=is_present,

        column_count=column_count,

        row_count=row_count,

    ))



status_df = spark.createDataFrame(status_rows)

columns_df = spark.createDataFrame(column_rows) if column_rows else spark.createDataFrame([], "object_name string, column_ordinal int, column_name string, data_type string, nullable boolean")



write_single_csv(status_df.orderBy("object_name"), f"{manifest_root}/gold_object_status")

write_single_csv(columns_df.orderBy("object_name", "column_ordinal"), f"{manifest_root}/gold_column_manifest")



if missing_required:

    raise ValueError(f"Missing required Gold objects: {missing_required}")



display(status_df.orderBy("object_name"))


In [ ]:
export_rows = []



for item in resolved_gold_objects:

    source_name = item.get("resolved_object_name") or item["object_name"]

    table_name = qualified_name(source_name)

    df = read_table_safe(table_name)

    if df is None:

        export_rows.append(Row(object_name=item["object_name"], source_table=table_name, exported=False, output_path=None, reason="missing source table"))

        continue



    relative_path = f"{csv_root}/{item['load_order']:02d}_{item['object_name']}"

    output_path = write_single_csv(df, relative_path)

    export_rows.append(Row(object_name=item["object_name"], source_table=table_name, exported=True, output_path=output_path, reason=None))



export_df = spark.createDataFrame(export_rows)

write_single_csv(export_df.orderBy("object_name"), f"{manifest_root}/gold_export_results")

display(export_df.orderBy("object_name"))


In [ ]:
if deploy_to_target:



    target_info = ensure_target_lakehouse()



    target_workspace_id_resolved = target_info["workspaceId"]



    target_lakehouse_name_resolved = target_info["displayName"]







    if target_workspace_id_resolved != source_workspace_id:



        raise ValueError(



            "This notebook can create the target lakehouse in another workspace, but Spark table writes are only implemented safely for the current workspace context. "



            "To deploy cross-workspace, copy the package to the target workspace and run this same notebook there."



        )







    deployment_rows = []



    for item in resolved_gold_objects:



        source_name = item.get("resolved_object_name") or item["object_name"]

        source_table = qualified_name(source_name)



        target_table = qualified_name(item["object_name"], lakehouse_name=target_lakehouse_name_resolved, schema_name=target_schema)



        df = read_table_safe(source_table)



        if df is None:



            deployment_rows.append(Row(object_name=item["object_name"], deployed=False, target_table=target_table, reason="missing source table"))



            continue







        df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(target_table)



        deployment_rows.append(Row(



            object_name=item["object_name"],



            deployed=True,



            target_table=target_table,



            reason=None,



        ))







    deployment_df = spark.createDataFrame(deployment_rows)



    deployment_context_df = spark.createDataFrame([Row(



        target_workspace_name=target_workspace_name,



        target_workspace_id=target_workspace_id_resolved,



        target_lakehouse_name=target_lakehouse_name_resolved,



        target_lakehouse_id=target_info["id"],



        target_lakehouse_created=target_info["created"],



        target_lakehouse_validated_by=target_info["validatedBy"],



    )])







    deployed_successfully = {row["object_name"] for row in deployment_rows if row["deployed"]}



    missing_for_ontology = sorted(list(set(required_for_ontology) - deployed_successfully))



    if missing_for_ontology:



        raise ValueError(



            f"Target hydration incomplete for ontology bootstrap. Missing required Delta tables: {missing_for_ontology}"



        )







    ontology_ready_df = spark.createDataFrame([Row(



        ontology_name=resolved_ontology_name,



        ontology_id=resolved_ontology_id,



        target_workspace_id=target_workspace_id_resolved,



        target_lakehouse_name=target_lakehouse_name_resolved,



        target_lakehouse_id=target_info["id"],



        target_sql_endpoint_hint="Use lakehouse SQL endpoint generated by Fabric for ontology binding",



        required_tables=",".join(required_for_ontology),



        hydration_status="ready",



    )])







    write_single_csv(deployment_context_df, f"{manifest_root}/gold_deployment_target")



    write_single_csv(deployment_df.orderBy("object_name"), f"{manifest_root}/gold_deployment_results")



    write_single_csv(ontology_ready_df, f"{manifest_root}/ontology_target_binding_hint")



    display(deployment_context_df)



    display(deployment_df.orderBy("object_name"))



    display(ontology_ready_df)



else:



    print("deploy_to_target=False, skipping target hydration.")


In [ ]:
if restore_ontology_to_target:

    if not deploy_to_target:

        raise ValueError("restore_ontology_to_target=True requires deploy_to_target=True so target context is fully resolved.")



    token = get_fabric_token()

    if not token:

        raise ValueError(

            "A Fabric token is required to restore ontology. Set fabric_token_override or run with notebookutils token support."

        )



    target_info_for_ontology = ensure_target_lakehouse()

    target_workspace_id_for_ontology = target_info_for_ontology["workspaceId"]

    target_lakehouse_id_for_ontology = target_info_for_ontology["id"]



    if not target_lakehouse_id_for_ontology:

        raise ValueError("Target lakehouse id is required to rewrite ontology bindings.")



    def _post_fabric(url: str, payload: dict):

        response = requests.post(url, headers=fabric_headers(token), data=json.dumps(payload), timeout=120)

        if response.status_code in (200, 201):

            if response.text:

                return response.json()

            return {}

        if response.status_code == 202:

            location = response.headers.get("Location") or response.headers.get("location")

            if not location:

                raise ValueError(f"LRO accepted without Location header for {url}")

            for _ in range(120):

                poll = requests.get(location, headers=fabric_headers(token), timeout=120)

                poll.raise_for_status()

                status_body = poll.json()

                status_value = status_body.get("status")

                if status_value == "Succeeded":

                    result_url = f"{location}/result"

                    result_resp = requests.get(result_url, headers=fabric_headers(token), timeout=120)

                    if result_resp.status_code == 200 and result_resp.text:

                        return result_resp.json()

                    return status_body

                if status_value in ("Failed", "Canceled"):

                    raise ValueError(f"Fabric LRO failed for {url}: {status_body}")

            raise TimeoutError(f"Fabric LRO timeout for {url}")

        response.raise_for_status()

        return response.json() if response.text else {}



    def _decode_definition_parts(parts):

        decoded = []

        for part in parts:

            payload = part.get("payload")

            if payload is None:

                continue

            decoded.append({

                "path": part.get("path"),

                "payloadType": part.get("payloadType", "InlineBase64"),

                "text": bytes.fromhex("").decode() if False else __import__("base64").b64decode(payload).decode("utf-8"),

            })

        return decoded



    def _encode_definition_parts(decoded_parts):

        out = []

        for part in decoded_parts:

            encoded_payload = __import__("base64").b64encode(part["text"].encode("utf-8")).decode("utf-8")

            out.append({

                "path": part["path"],

                "payload": encoded_payload,

                "payloadType": part.get("payloadType", "InlineBase64"),

            })

        return out



    def _rewrite_binding_refs(obj):

        if isinstance(obj, dict):

            rewritten = {}

            for k, v in obj.items():

                if k == "workspaceId" and v == source_workspace_id:

                    rewritten[k] = target_workspace_id_for_ontology

                elif k == "itemId" and v == source_gold_lakehouse_id:

                    rewritten[k] = target_lakehouse_id_for_ontology

                else:

                    rewritten[k] = _rewrite_binding_refs(v)

            return rewritten

        if isinstance(obj, list):

            return [_rewrite_binding_refs(v) for v in obj]

        return obj



    def _get_source_ontology_definition():

        url = f"https://api.fabric.microsoft.com/v1/workspaces/{source_workspace_id}/items/{resolved_ontology_id}/getDefinition"

        response = requests.post(url, headers=fabric_headers(token), data=json.dumps({}), timeout=120)

        if response.status_code == 200:

            body = response.json()

            if "definition" in body:

                return body["definition"]

            raise ValueError(f"Unexpected getDefinition payload: {body}")

        if response.status_code == 202:

            location = response.headers.get("Location") or response.headers.get("location")

            if not location:

                raise ValueError("getDefinition LRO missing Location header")

            for _ in range(120):

                poll = requests.get(location, headers=fabric_headers(token), timeout=120)

                poll.raise_for_status()

                status_body = poll.json()

                if status_body.get("status") == "Succeeded":

                    result_resp = requests.get(f"{location}/result", headers=fabric_headers(token), timeout=120)

                    result_resp.raise_for_status()

                    result_body = result_resp.json()

                    return result_body["definition"]

                if status_body.get("status") in ("Failed", "Canceled"):

                    raise ValueError(f"getDefinition failed: {status_body}")

            raise TimeoutError("getDefinition polling timeout")

        response.raise_for_status()

        raise ValueError("Unable to fetch ontology definition")



    source_definition = _get_source_ontology_definition()

    decoded_parts = _decode_definition_parts(source_definition.get("parts", []))



    rewritten_parts = []

    for part in decoded_parts:

        text = part["text"]

        path = part["path"] or ""

        if path.endswith(".json"):

            try:

                obj = json.loads(text)

                obj = _rewrite_binding_refs(obj)

                text = json.dumps(obj, ensure_ascii=False, indent=2)

            except Exception:

                pass

        rewritten_parts.append({"path": part["path"], "payloadType": part.get("payloadType", "InlineBase64"), "text": text})



    rewritten_definition = {"parts": _encode_definition_parts(rewritten_parts)}



    workspace_items = list_workspace_items(target_workspace_id_for_ontology, token)

    target_ontology = None

    if target_ontology_id:

        target_ontology = next((i for i in workspace_items if i.get("id") == target_ontology_id), None)

    if not target_ontology:

        target_ontology = next((i for i in workspace_items if i.get("type") == "Ontology" and i.get("displayName") == target_ontology_name), None)



    created_ontology = False

    if not target_ontology:

        create_payload = {"displayName": target_ontology_name, "type": "Ontology"}

        created = _post_fabric(

            f"https://api.fabric.microsoft.com/v1/workspaces/{target_workspace_id_for_ontology}/items",

            create_payload,

        )

        created_id = created.get("id")

        if not created_id:

            workspace_items = list_workspace_items(target_workspace_id_for_ontology, token)

            target_ontology = next((i for i in workspace_items if i.get("type") == "Ontology" and i.get("displayName") == target_ontology_name), None)

            if not target_ontology:

                raise ValueError("Ontology creation response did not return an id and item lookup failed.")

        else:

            target_ontology = {"id": created_id, "displayName": target_ontology_name, "type": "Ontology"}

        created_ontology = True



    if (not created_ontology) and (not overwrite_target_ontology_definition):

        raise ValueError(

            "Target ontology already exists and overwrite_target_ontology_definition=False. "

            "Set overwrite_target_ontology_definition=True to apply source definition and bindings."

        )



    update_payload = {"definition": rewritten_definition}

    _post_fabric(

        f"https://api.fabric.microsoft.com/v1/workspaces/{target_workspace_id_for_ontology}/items/{target_ontology['id']}/updateDefinition",

        update_payload,

    )



    ontology_restore_df = spark.createDataFrame([Row(

        source_ontology_id=resolved_ontology_id,

        source_ontology_name=resolved_ontology_name,

        target_workspace_id=target_workspace_id_for_ontology,

        target_ontology_id=target_ontology["id"],

        target_ontology_name=target_ontology_name,

        target_lakehouse_id=target_lakehouse_id_for_ontology,

        target_lakehouse_name=target_info_for_ontology["displayName"],

        created=created_ontology,

        overwrite_applied=True,

        status="ready",

    )])



    write_single_csv(ontology_restore_df, f"{manifest_root}/ontology_restore_results")

    display(ontology_restore_df)

else:

    print("restore_ontology_to_target=False, skipping ontology restore step.")


## Outputs



This notebook writes a portable package under `/lakehouse/default/Files/github_portable/gold/` with:



- `manifests/package_context`

- `manifests/ontology_scope`

- `manifests/gold_objects`

- `manifests/gold_notebooks`

- `manifests/gold_object_status`

- `manifests/gold_column_manifest`

- `manifests/gold_export_results`

- `manifests/gold_deployment_target` when deployment is enabled

- `manifests/gold_deployment_results` when deployment is enabled

- `manifests/ontology_target_binding_hint` when deployment is enabled and required tables are hydrated

- `manifests/ontology_restore_results` when ontology restoration is enabled

- `csv/<load_order>_<object_name>` for every exported Gold table



Target configured in this notebook:



- workspace: LoanDemoEnv

- lakehouse: Gold_AnalyzeLoanV3_Target

- lakehouse id: 49b3f9e2-0aa8-457c-bd6f-ea87a2cf15e3

- ontology restore: enabled by `restore_ontology_to_target`



When ontology restore is enabled, the notebook:



1. Reads the source ontology definition (`AnalyzeLoanOnV3`).

2. Rewrites DataBindings and relationship contextualization references from source Gold to target Gold.

3. Creates the ontology in the target workspace if missing.

4. Updates ontology definition so entities, nodes, relationships and bindings are ready for querying.
